# 11 · FX Spot, Swaps, and Options

FX instruments combine discount curves, FX matrices, and volatility surfaces to price spot exchanges, swaps, and options.

### Learning Objectives
- Build FX market data (discount curves per currency, FX matrix, vol surface)
- Price spot exchanges and simple FX swaps
- Value European FX options with delta/gamma reporting

In [ ]:
from datetime import date, timedelta

from finstack import Money
from finstack.core.currency import USD, EUR, JPY
from finstack.core.market_data import MarketContext
from finstack.core.market_data.fx import FxMatrix
from finstack.core.market_data.surfaces import VolSurface
from finstack.core.market_data.term_structures import DiscountCurve
from finstack.valuations.instruments import FxSpot, FxSwap, FxOption
from finstack.valuations.pricer import create_standard_registry

as_of = date(2024, 1, 2)
market = MarketContext()
market.insert_discount(
    DiscountCurve(
        "USD-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9975), (1.0, 0.9945), (3.0, 0.9720), (5.0, 0.9450)],
    )
)
market.insert_discount(
    DiscountCurve(
        "EUR-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9980), (1.0, 0.9960), (3.0, 0.9800), (5.0, 0.9550)],
    )
)
market.insert_discount(
    DiscountCurve(
        "JPY-OIS",
        as_of,
        [(0.0, 1.0), (0.5, 0.9990), (1.0, 0.9980), (3.0, 0.9920), (5.0, 0.9850)],
    )
)
fx_matrix = FxMatrix()
fx_matrix.set_quote(EUR, USD, 1.0850)
fx_matrix.set_quote(USD, JPY, 148.50)
fx_matrix.set_quote(EUR, JPY, 161.20)
market.insert_fx(fx_matrix)
market.insert_surface(
    VolSurface(
        "EURUSD-VOL",
        expiries=[0.25, 0.5, 1.0, 2.0],
        strikes=[1.05, 1.10, 1.15],
        grid=[
            [0.14, 0.13, 0.12],
            [0.13, 0.12, 0.11],
            [0.12, 0.11, 0.10],
            [0.11, 0.10, 0.095],
        ],
    )
)
registry = create_standard_registry()


## 1. Spot Exchange
`FxSpot.create` exchanges notionals on settlement date. Provide spot rate, notional, and settlement lag.

In [ ]:
spot = FxSpot.create(
    "EURUSD-SPOT",
    EUR,
    USD,
    settlement=as_of + timedelta(days=2),
    spot_rate=1.0860,
    notional=Money(1_000_000, EUR),
)
spot_res = registry.price(spot, "discounting", market)
print("Spot PV:", spot_res.value)


## 2. FX Swap
Swaps exchange notionals at near/far dates with quoted forward points. Provide discount curve IDs per leg.

In [ ]:
near = as_of + timedelta(days=2)
far = near + timedelta(days=180)
fx_swap = FxSwap.create(
    "EURUSD-SWAP",
    EUR,
    USD,
    Money(5_000_000, EUR),
    near,
    far,
    discount_curve_domestic="USD-OIS",
    discount_curve_foreign="EUR-OIS",
    near_rate=1.0865,
    far_rate=1.0920,
)
swap_res = registry.price_with_metrics(fx_swap, "discounting", market, ["carry_pv"])
print("FX swap PV:", swap_res.value)
print("Carry PV:", swap_res.measures.get("carry_pv"))


## 3. European FX Option
FX options reference the FX vol surface and support delta/gamma metrics.

In [ ]:
fx_call = FxOption.european_call(
    "EURUSD-CALL-1Y",
    EUR,
    USD,
    strike=1.10,
    expiry=date(2025, 1, 2),
    notional=Money(2_000_000, EUR),
    vol_surface="EURUSD-VOL",
)
fx_res = registry.price_with_metrics(fx_call, "discounting", market, ["delta", "gamma"])
print("FX option PV:", fx_res.value)
print("Delta:", fx_res.measures.get("delta"))
print("Gamma:", fx_res.measures.get("gamma"))


## Summary
- Discount curves per currency plus an `FxMatrix` create a consistent FX market.
- Spot and swap trades require settlement dates and forward quotes.
- FX options leverage the same infrastructure plus vol surfaces to produce Greeks.